# Physics-informed fine-tuning of the PhysicsNeMo Stokes surrogate on a Kaggle GPU

Launcher for [physicsnemo-stokes-finetuning](https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning). It checks out one commit of the repository and calls its scripts; it contains no model or training code.

Session settings: **Accelerator** an NVIDIA GPU, **Internet** on.

| Step | What runs | When |
|---|---|---|
| A | install, dataset download with SHA-256 check, seeded split, tests | always |
| B | CUDA smoke study (both stages, tiny) | always |
| C0 | device checks: resume after an interruption, repeatability, checkpoint round trip, gradients | always |
| C | timing and memory probe: one epoch of the data-driven stage, 200 fine-tuning iterations | always |
| D | full study, `configs/full.yaml`: 500 epochs on 500 meshes, then 10000 fine-tuning iterations on each of 10 test meshes | `RUN_FULL = True` |

Before D the notebook projects the runtime from the probe and compares it with what is left of `SESSION_BUDGET_HOURS`. If the projection does not fit, the step is **not started and nothing is reduced**; the notebook reports the numbers and finishes.

There is no separate `RUN_OFFICIAL` step. `configs/official.yaml` holds NVIDIA's settings and fine-tunes test mesh 7 only, as the upstream script does; `configs/full.yaml` uses the same settings and seed for both stages and fine-tunes test meshes 0 to 9, so the upstream case is sample 7 of step D.

In [ ]:
REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning.git"
REF = "71e7c79268006ad1396c9effb6f12dd0acae67a8"   # output of `git rev-parse HEAD` for the commit to run
RUN_FULL = False                              # step D: only the value True starts it
SESSION_BUDGET_HOURS = 9.0                    # GPU time you are willing to spend in this session; check Kaggle's current limits
GITHUB_TOKEN_SECRET = None                    # name of a Kaggle secret holding a GitHub token; only for a private repository

import re, time
T0 = time.time()
assert re.fullmatch(r"[0-9a-f]{40}", REF), "REF must be a full 40-character commit SHA, not a branch or tag."
WORK = "/kaggle/working"
REPO = f"{WORK}/physicsnemo-stokes-finetuning"

def hours_left():
    return SESSION_BUDGET_HOURS - (time.time() - T0) / 3600

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "CUDA is not available. Select a GPU accelerator; this notebook does not fall back to CPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
import os, subprocess
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.run(["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True).stdout.strip()

if not os.path.isdir(REPO):
    url = REPO_URL
    if GITHUB_TOKEN_SECRET:
        from kaggle_secrets import UserSecretsClient
        url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET) + "@")
    subprocess.run(["git", "clone", "--quiet", url, REPO], check=True)   # not echoed: the URL may hold a token
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO, check=True)
assert tree_is_clean(), "the existing working tree is not clean"
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True).stdout.strip()
assert head == REF, f"checked out {head}, expected {REF}"
assert tree_is_clean(), "the working tree is not clean"
print("commit", head)

## Step A: environment, dataset, tests

The requirements pin PhysicsNeMo to a commit of its main branch, which needs `torch>=2.13`; `pip` may replace the PyTorch build of the image. `torch_scatter` has no prebuilt wheel for that PyTorch and is compiled from source (CPU sources only; the sum aggregation of MeshGraphNet runs through `Tensor.scatter_add_` on the GPU). The versions are printed afterwards in a new process, and CUDA is checked again.

In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh("FORCE_ONLY_CPU=1 pip install -q --no-build-isolation --no-binary torch-scatter torch-scatter==2.1.2", cwd=REPO)
# The Kaggle image preinstalls RAPIDS (cuml, cudf, cupy, rmm, ...) and numba. PhysicsNeMo imports cuml
# whenever it is installed; cudf then imports numba, which rejects the NumPy that requirements.txt
# installs (first launch, 2026-11-07: "Numba needs NumPy 2.2 or less. Got NumPy 2.5"). The validated
# environment (requirements.lock.txt) contains none of these packages, so they are removed from this
# session instead of downgrading NumPy.
RAPIDS_IMPORTS = ("cuml", "cudf", "dask_cudf", "cugraph", "pylibcugraph", "pylibcugraphops", "nx_cugraph", "cuspatial",
                  "cuvs", "cuxfilter", "cucim", "rmm", "pylibraft", "raft_dask", "cupy", "numba", "numba_cuda")
sh("python -c \"from importlib.metadata import packages_distributions as p; m = p(); "
   f"print(' '.join(sorted({{d for name in {RAPIDS_IMPORTS!r} for d in m.get(name, [])}})))\" > /tmp/rapids_dists.txt")
dists = open("/tmp/rapids_dists.txt").read().split()
print("removing from this session:", dists)
if dists:
    sh("pip uninstall -y -q " + " ".join(dists))
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
# every project module imports on this environment before any data or GPU time is spent
sh('PYTHONPATH=src python -c "import numpy, stokes_ft.data, stokes_ft.engine, stokes_ft.model, stokes_ft.study, '
   "stokes_ft.physics, stokes_ft.weak, stokes_ft.variants, stokes_ft.fem; print('project imports OK | numpy', numpy.__version__)\"", cwd=REPO)

In [ ]:
sh("python scripts/prepare_data.py", cwd=REPO)      # 172 MB from NGC, SHA-256 verified, 800/100/100 split with seed 0
assert tree_is_clean(), "preparing the data modified the clone"
sh("python -m pytest -q", cwd=REPO)

## Step B: CUDA smoke study

Six training meshes, two epochs, twenty fine-tuning iterations on two test meshes. The numbers are not results.

In [ ]:
sh("python scripts/run_study.py --config smoke device=cuda name=smoke_cuda", cwd=REPO)
sh("python scripts/plot_results.py --study runs/smoke_cuda --out runs/smoke_cuda/figures", cwd=REPO)

## Step C0: device checks

Resume after an interruption (one epoch, then resumed to two, against an uninterrupted two-epoch run), repeatability of two runs with the same seed, checkpoint round trip, and finite gradients, on the smoke configuration. `REF` may predate `src/stokes_ft/device_checks.py`; the cell therefore writes a verbatim copy of that module (checked against the repository by `tests/test_notebook.py`) to `/kaggle/working`, outside the clone, and runs it against the checked-out package. The step stops the notebook if a check fails.

In [ ]:
# verbatim copy of src/stokes_ft/device_checks.py; relative imports resolved against the checked-out package
DEVICE_CHECKS_SOURCE = r'''"""Checks of the training pipeline on one device, run before a long study (``scripts/device_checks.py``).

* **Resume after an interruption**: a run trained for one epoch and resumed to two must match an
  uninterrupted two-epoch run to rounding (checkpoint of model, optimiser, scheduler; seeded order).
* **Repeatability**: two uninterrupted runs with the same seed must agree to rounding (scatter-add
  accumulation order is not fixed, so they are not required to be bit-for-bit identical).
* **Checkpoint round trip**: two models loaded from the same checkpoint have bit-identical weights,
  and their outputs agree to rounding (``OUTPUT_TOLERANCE``; scatter-add is not bit-reproducible on
  CUDA, so even one model called twice differs in the last bits; that difference is reported).
* **Gradients**: one backward pass from the loaded model gives finite, nonzero gradients for every
  parameter that requires one.

The learning-rate decay is set explicitly so that the one-epoch and two-epoch runs use the same
schedule (the default decay depends on the number of epochs). The numbers verify the pipeline;
they are not results.
"""

import csv
import shutil
from pathlib import Path

import numpy as np
import torch
from physicsnemo.utils import load_checkpoint

from .data import make_dataset, prepare_split
from .engine import train_data_driven
from .model import build_meshgraphnet

TOLERANCE = 1e-3
OUTPUT_TOLERANCE = 1e-4  # max |difference| / max |output| between two loads of one checkpoint


def _history(run_dir: Path) -> dict:
    with open(run_dir / "history.csv") as f:
        rows = list(csv.DictReader(f))
    return {k: np.array([float(r[k]) for r in rows]) for k in ("loss", "val_rel_l2_u", "val_rel_l2_v", "val_rel_l2_p")}


def _max_rel_diff(a: dict, b: dict) -> float:
    return float(max(np.max(np.abs(a[k] - b[k]) / np.maximum(np.abs(b[k]), 1e-30)) for k in b))


def run_device_checks(cfg, out_dir: Path, device: torch.device, log=print) -> dict:
    out_dir = Path(out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    prepare_split(cfg)
    two = cfg.copy()
    two.epochs, two.lr_decay_rate = 2, 0.99
    two.training.checkpoint_every = 1
    one = two.copy()
    one.epochs = 1

    interrupted, uninterrupted, repeat = out_dir / "interrupted", out_dir / "uninterrupted", out_dir / "repeat"
    first = train_data_driven(one, interrupted, device, log=log)
    resumed = train_data_driven(two, interrupted, device, resume=True, log=log)
    reference = train_data_driven(two, uninterrupted, device, log=log)
    train_data_driven(two, repeat, device, log=log)
    h_int, h_ref, h_rep = _history(interrupted), _history(uninterrupted), _history(repeat)

    def load():
        model = build_meshgraphnet(two).to(device)
        epoch = load_checkpoint(str(uninterrupted / "checkpoints" / "data_driven"), models=model, device=device)
        return model, epoch

    a, epoch = load()
    b, _ = load()
    graph = next(iter(make_dataset(two, "train", uninterrupted))).to(device)
    a.eval(), b.eval()
    sa, sb = a.state_dict(), b.state_dict()
    weights_identical = sa.keys() == sb.keys() and all(torch.equal(sa[k], sb[k]) for k in sa)
    with torch.no_grad():
        ya, ya2, yb = (m(graph.x, graph.edge_attr, graph) for m in (a, a, b))
    # scatter-add accumulation order is not fixed on CUDA: even one model called twice is not
    # bit-identical, so outputs are compared to rounding and the difference is reported
    scale = float(ya.abs().max())
    output_diff = float((ya - yb).abs().max()) / scale
    repeat_call_diff = float((ya - ya2).abs().max()) / scale
    identical = weights_identical and output_diff < OUTPUT_TOLERANCE
    a.train()
    loss = torch.nn.functional.mse_loss(a(graph.x, graph.edge_attr, graph), graph.y)
    loss.backward()
    grads = [p.grad for p in a.parameters() if p.requires_grad]
    grads_finite = all(g is not None and bool(torch.isfinite(g).all()) for g in grads)
    grad_norm = float(torch.sqrt(sum((g.double() ** 2).sum() for g in grads if g is not None)))

    finite = all(np.isfinite(v).all() for h in (h_int, h_ref, h_rep) for v in h.values())
    out = {
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "epochs": 2, "lr_decay_rate": 0.99, "tolerance": TOLERANCE,
        "interrupted_after_epoch": first["epochs"], "resumed_flag": resumed["resumed"],
        "resume_epochs_in_history": len(h_int["loss"]),
        "resume_max_relative_difference": _max_rel_diff(h_int, h_ref),
        "repeat_max_relative_difference": _max_rel_diff(h_rep, h_ref),
        "checkpoint_epoch": epoch, "checkpoint_weights_identical": weights_identical,
        "checkpoint_output_max_relative_difference": output_diff,
        "same_model_repeat_call_max_relative_difference": repeat_call_diff,
        "checkpoint_round_trip_ok": identical,
        "gradients_finite": grads_finite, "gradient_norm": grad_norm,
        "all_losses_finite": finite,
        "losses": {k: v.tolist() for k, v in h_ref.items()},
        "peak_cuda_memory_mb": reference["peak_cuda_memory_mb"],
    }
    out["passed"] = bool(
        out["resumed_flag"] and out["resume_epochs_in_history"] == 2 and out["checkpoint_epoch"] == 2
        and out["resume_max_relative_difference"] < TOLERANCE and out["repeat_max_relative_difference"] < TOLERANCE
        and identical and grads_finite and grad_norm > 0 and finite
    )
    return out
'''
import pathlib
checks_dir = pathlib.Path(WORK) / "device_checks_module"
checks_dir.mkdir(exist_ok=True)
(checks_dir / "stokes_device_checks.py").write_text(DEVICE_CHECKS_SOURCE.replace("from .", "from stokes_ft."))
(checks_dir / "run_device_checks.py").write_text(
    "import json, sys\n"
    "import torch\n"
    "from stokes_ft.config import load_config\n"
    "from stokes_ft.provenance import collect_metadata, write_json\n"
    "from stokes_device_checks import run_device_checks\n"
    "cfg = load_config('smoke', ['device=cuda'])\n"
    "device = torch.device('cuda')\n"
    "out = run_device_checks(cfg, 'runs/device_checks/runs', device)\n"
    "out['environment'] = collect_metadata(device)\n"
    "write_json('runs/device_checks/device_checks.json', out)\n"
    "print(json.dumps({k: v for k, v in out.items() if k not in ('losses', 'environment')}, indent=1))\n"
    "sys.exit(0 if out['passed'] else 1)\n")
sh(f"PYTHONPATH={checks_dir}:{REPO}/src python {checks_dir}/run_device_checks.py", cwd=REPO)
assert tree_is_clean(), "the device checks modified the clone"
sh(f"cp runs/device_checks/device_checks.json {WORK}/physicsnemo-stokes-finetuning-device-checks.json", cwd=REPO)
print("DEVICE CHECKS PASSED")


## Step C: timing and memory probe

One epoch of the data-driven stage of `configs/full.yaml` (500 meshes) and 200 fine-tuning iterations on test mesh 7. The projection scales these two measurements to 500 epochs and to 10 meshes of 10000 iterations. It is an extrapolation and includes start-up cost, so it tends to overestimate.

In [ ]:
import json
sh('python scripts/run_study.py --config full name=full_probe epochs=1 pi_iters=200 "finetune.samples=[7]" '
   '"evaluation.plot_samples=[7]" training.checkpoint_every=1', cwd=REPO)
train = json.load(open(f"{REPO}/runs/full_probe/train_metrics.json"))
tune = json.load(open(f"{REPO}/runs/full_probe/finetune/finetune_7.json"))
meta = json.load(open(f"{REPO}/runs/full_probe/study_metadata.json"))["sessions"][-1]
train_h = train["seconds_per_iteration"] * 500 * 500 / 3600      # 500 epochs x 500 meshes in configs/full.yaml
tune_h = tune["seconds_per_iteration"] * 10000 * 10 / 3600       # 10000 iterations x 10 test meshes in configs/full.yaml
full_projection_h = train_h + tune_h
print(f"GPU: {meta['gpu_name']} | torch {meta['torch_version']} | CUDA {meta['cuda_version']}")
print(f"data-driven stage: {train['seconds_per_iteration'] * 1e3:.1f} ms per iteration, peak {train['peak_cuda_memory_mb']:.0f} MB -> {train_h:.2f} h for 250000 iterations")
print(f"fine-tuning stage: {tune['seconds_per_iteration'] * 1e3:.1f} ms per iteration, peak {tune['peak_cuda_memory_mb']:.0f} MB -> {tune_h:.2f} h for 10 meshes")
print(f"PROJECTION for step D: {full_projection_h:.1f} h. Left of the session budget: {hours_left():.1f} h.")
import pathlib
ckpt_mb = sum(p.stat().st_size for p in pathlib.Path(f"{REPO}/runs/full_probe/checkpoints").rglob("*") if p.is_file()) / 2**20
print(f"checkpoints kept by the probe (latest epoch only, plus one fine-tuned network): {ckpt_mb:.1f} MB")
# Preserve the probe evidence even when the full study is disabled or exceeds the budget.
for study in ("smoke_cuda", "full_probe"):
    sh(f"python scripts/results.py package runs/{study}", cwd=REPO)
    sh(f"cp runs/{study}.zip {WORK}/physicsnemo-stokes-finetuning-{study}.zip", cwd=REPO)


## Step D: full study (`configs/full.yaml`)

The MeshGraphNet checkpoint before the physics-informed stage and the ten fine-tuning networks after it are kept in `runs/full/checkpoints` and stored in the checkpoint archive. Both are evaluated on the same ten test meshes.

In [ ]:
full_done = False
if RUN_FULL is not True:
    print("Step D skipped: RUN_FULL is False.")
elif full_projection_h > hours_left():
    print(f"STEP D NOT STARTED: projected {full_projection_h:.1f} h, {hours_left():.1f} h left of SESSION_BUDGET_HOURS = {SESSION_BUDGET_HOURS}.")
    print("The configuration was not reduced. Decide how to proceed before changing configs/full.yaml.")
else:
    sh("python scripts/run_study.py --config full --resume", cwd=REPO)
    sh("python scripts/plot_results.py --study runs/full --out runs/full/figures", cwd=REPO)
    sh("python scripts/results.py package runs/full", cwd=REPO)
    sh("python scripts/results.py bundle runs/full", cwd=REPO)
    sh(f"cp runs/full.zip {WORK}/physicsnemo-stokes-finetuning-full.zip && cp runs/full_checkpoints.zip {WORK}/physicsnemo-stokes-finetuning-full-checkpoints.zip", cwd=REPO)
    sh(f"cd runs/full && zip -q -r {WORK}/physicsnemo-stokes-finetuning-full-figures.zip figures", cwd=REPO)
    full_done = True
    print("STEP D COMPLETE.")

In [ ]:
import glob
print("Archives in /kaggle/working:")
for path in sorted(glob.glob(f"{WORK}/physicsnemo-stokes-finetuning-*.zip")):
    print(f"  {path}  {os.path.getsize(path) / 2**20:.1f} MB")
print(f"Elapsed: {(time.time() - T0) / 3600:.2f} h. Steps A to C: done. Step D: {'done' if full_done else 'not run'}.")
if not full_done:
    print("KAGGLE CUDA CHECKS COMPLETE. No full study was run.")